In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, applications
from sklearn.utils import class_weight
import numpy as np
import os

# --- STEP 1: PATH SETUP ---
# Kaggle automatically converts notebook names to lowercase for the input folder.
# We expect the folder to be 'preprocessingdata' based on your notebook name.
INPUT_ROOT = '/kaggle/input/preprocessingdata' 
DATA_DIR = os.path.join(INPUT_ROOT, 'spectrogram_dataset')

print("Checking for dataset...")
if os.path.exists(DATA_DIR):
    print(f"✅ Success! Found dataset at: {DATA_DIR}")
    print("Folders found:", os.listdir(DATA_DIR))
else:
    print(f"❌ Error: Could not find dataset at {DATA_DIR}")
    print("Available folders in input:", os.listdir('/kaggle/input'))
    print("Please update the 'INPUT_ROOT' variable above with the correct folder name from the list.")

# --- CONFIGURATION ---
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
EPOCHS = 10  # You can increase this to 15 or 20 for better results

# --- STEP 2: LOAD DATA ---
print("\n--- Loading Data ---")
# Training Data
train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, 'train'),
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

# Validation Data
val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, 'val'),
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Print Class Mapping
class_names = train_ds.class_names
print(f"Class Mapping: {class_names}") 
# usually ['fake', 'real'] -> 0 is fake, 1 is real (alphabetical)

# --- STEP 3: HANDLE IMBALANCE (Auto-Weight Calculation) ---
print("\n--- Calculating Class Weights ---")
# We need to loop through the data once to count the classes
train_labels = []
print("Counting classes (this takes a few seconds)...")
for images, labels in train_ds.unbatch():
    train_labels.append(labels.numpy())

# Compute weights
# This formula makes the model pay more attention to the minority class
class_weights_vals = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

# Convert to dictionary for Keras
class_weights_dict = dict(enumerate(class_weights_vals))
print(f"Calculated Weights: {class_weights_dict}")
print("(The higher number is the class the model will focus on more)")

# --- STEP 4: BUILD MODEL (Transfer Learning) ---
print("\n--- Building Model ---")
# Using MobileNetV2 (Lightweight and Fast)
base_model = applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False, # Remove the original classification layer
    weights='imagenet'
)
base_model.trainable = False # Freeze base model so we don't destroy pretrained patterns

model = models.Sequential([
    # Preprocessing: Scale pixel values to [-1, 1] required by MobileNetV2
    layers.Rescaling(1./127.5, offset=-1, input_shape=IMG_SIZE + (3,)),
    
    base_model,
    
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3), # Dropout helps prevent overfitting
    
    # Final Layer: 1 neuron with Sigmoid (Output is probability between 0 and 1)
    layers.Dense(1, activation='sigmoid') 
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# --- STEP 5: TRAIN ---
print("\n--- Starting Training ---")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights_dict # <--- This applies the imbalance fix
)

# --- STEP 6: SAVE MODEL ---
save_path = '/kaggle/working/audio_detector.h5'
model.save(save_path)
print(f"\n✅ Model saved successfully at: {save_path}")
print("You can now download this file from the 'Output' tab of this notebook.")

2025-11-24 13:28:58.108166: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763990938.345153      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763990938.414800      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

/usr/local/lib/python3.11/dist-packages/keras/src/layers/preprocessing/tf_data_layer.py:19: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
I0000 00:00:1763991013.845910     112 service.cc:148] XLA service 0x7d5234003840 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1763991013.846629     112 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1763991013.846651     112 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1763991014.850559     112 cuda_dnn.cc:529] Loaded cuDNN version 90300


  7/696 ━━━━━━━━━━━━━━━━━━━━ 17s 26ms/step - accuracy: 0.3820 - loss: 0.8126

I0000 00:00:1763991019.252559     112 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


696/696 ━━━━━━━━━━━━━━━━━━━━ 43s 47ms/step - accuracy: 0.8658 - loss: 0.3051 - val_accuracy: 0.9644 - val_loss: 0.1069
Epoch 2/10
696/696 ━━━━━━━━━━━━━━━━━━━━ 23s 33ms/step - accuracy: 0.9566 - loss: 0.1265 - val_accuracy: 0.9696 - val_loss: 0.0896
Epoch 3/10
696/696 ━━━━━━━━━━━━━━━━━━━━ 23s 33ms/step - accuracy: 0.9632 - loss: 0.1084 - val_accuracy: 0.9718 - val_loss: 0.0782
Epoch 4/10
696/696 ━━━━━━━━━━━━━━━━━━━━ 23s 34ms/step - accuracy: 0.9643 - loss: 0.0995 - val_accuracy: 0.9736 - val_loss: 0.0743
Epoch 5/10
696/696 ━━━━━━━━━━━━━━━━━━━━ 24s 34ms/step - accuracy: 0.9670 - loss: 0.0954 - val_accuracy: 0.9737 - val_loss: 0.0717
Epoch 6/10
696/696 ━━━━━━━━━━━━━━━━━━━━ 23s 34ms/step - accuracy: 0.9666 - loss: 0.0950 - val_accuracy: 0.9759 - val_loss: 0.0669
Epoch 7/10
696/696 ━━━━━━━━━━━━━━━━━━━━ 24s 34ms/step - accuracy: 0.9670 - loss: 0.0919 - val_accuracy: 0.9756 - val_loss: 0.0648
Epoch 8/10
696/696 ━━━━━━━━━━━━━━━━━━━━ 24s 34ms/step - accuracy: 0.9692 - loss: 0.0907 - val_accurac


✅ Model saved successfully at: /kaggle/working/audio_detector.h5
You can now download this file from the 'Output' tab of this notebook.


In [ ]:
# --- STEP 1: INSTALL GRADIO ---
# Run this line first if you haven't installed it in this session
!pip install -q gradio

import gradio as gr
import tensorflow as tf
import numpy as np
import librosa
import cv2
import os

# --- STEP 2: LOAD THE TRAINED MODEL ---
MODEL_PATH = '/kaggle/working/audio_detector.h5'

# Check if model exists
if not os.path.exists(MODEL_PATH):
    print(f"❌ Error: Model not found at {MODEL_PATH}. Please make sure you finished training and saved the model.")
else:
    print("Loading model...")
    model = tf.keras.models.load_model(MODEL_PATH)
    print("✅ Model loaded successfully!")

# --- STEP 3: DEFINE PREPROCESSING (MUST MATCH TRAINING EXACTLY) ---
def preprocess_audio(audio_path):
    # 1. Settings (Must match what you used in Notebook 1)
    SAMPLE_RATE = 16000
    DURATION = 4
    N_MELS = 128
    
    try:
        # Load audio
        y, sr = librosa.load(audio_path, sr=SAMPLE_RATE, duration=DURATION)
        
        # Pad if too short
        target_length = DURATION * SAMPLE_RATE
        if len(y) < target_length:
            y = np.pad(y, (0, target_length - len(y)), mode='constant')
        elif len(y) > target_length:
            y = y[:target_length]
            
        # Create Spectrogram
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        
        # Normalize to 0-255 (Standard Image format)
        img = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min())
        img = (img * 255).astype(np.uint8)
        
        # Flip vertically (Low freq at bottom)
        img = np.flip(img, axis=0)
        
        # Resize to 224x224 (Input size of MobileNetV2)
        img_resized = cv2.resize(img, (224, 224))
        
        # Convert grayscale to RGB (3 channels) because MobileNet expects 3 channels
        img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_GRAY2RGB)
        
        # Add Batch Dimension (1, 224, 224, 3)
        img_batch = np.expand_dims(img_rgb, axis=0)
        
        return img_batch
        
    except Exception as e:
        print(f"Error processing audio: {e}")
        return None

# --- STEP 4: PREDICTION FUNCTION ---
def predict_deepfake(audio_file):
    # Preprocess the audio file path coming from Gradio
    processed_image = preprocess_audio(audio_file)
    
    if processed_image is None:
        return "Error processing file."
    
    # Predict
    prediction = model.predict(processed_image)
    score = prediction[0][0] # Get the number (0 to 1)
    
    # Interpret result
    # In your training: 0 was usually 'fake', 1 was 'real' (alphabetical order)
    # But check your class_names print output to be sure! 
    # Assuming: Fake=0, Real=1
    
    confidence = score if score > 0.5 else 1 - score
    label = "REAL (Human)" if score > 0.5 else "FAKE (AI Generated)"
    
    return f"{label} \nConfidence: {confidence:.2%}"

# --- STEP 5: LAUNCH INTERFACE ---
interface = gr.Interface(
    fn=predict_deepfake,
    inputs=gr.Audio(type="filepath", label="Upload Audio"),
    outputs="text",
    title="🎙️ Deepfake Audio Detector",
    description="Upload an audio file to check if it is Real Human Voice or AI Generated."
)

# Launch with share=True to get a public link
interface.launch(share=True, debug=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 11.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 44.1 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
Loading model...


✅ Model loaded successfully!
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://5257573674b7c6ce96.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
